In [1]:
# 导入依赖库
import os
import gzip
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# 定义MNIST数据集解析函数
def load_mnist(path, kind='train'):
    # 拼接图像/标签文件路径
    images_path = os.path.join(path, f'{kind}-images-idx3-ubyte.gz')
    labels_path = os.path.join(path, f'{kind}-labels-idx1-ubyte.gz')
    
    # 读取标签（跳过文件头）
    with gzip.open(labels_path, 'rb') as lbpath:
        labels = np.frombuffer(lbpath.read(), dtype=np.uint8, offset=8)
    
    # 读取图像并转为28×28一维特征
    with gzip.open(images_path, 'rb') as imgpath:
        images = np.frombuffer(imgpath.read(), dtype=np.uint8, offset=16)
        images = images.reshape(len(labels), 784)  # 784=28×28
    return images, labels

# 加载数据集（仓库相对路径）
mnist_path = 'data/data65/'
X_train, y_train = load_mnist(mnist_path, kind='train')  # 训练集（60000样本）
X_test, y_test = load_mnist(mnist_path, kind='t10k')     # 测试集（10000样本）

# 验证数据加载结果
print("=== 数据集加载完成 ===")
print(f"训练集规模：{X_train.shape[0]}个样本，特征维度：{X_train.shape[1]}")
print(f"测试集规模：{X_test.shape[0]}个样本，数字类别数：{len(np.unique(y_train))}")


=== 数据集加载完成 ===
训练集规模：60000个样本，特征维度：784
测试集规模：10000个样本，数字类别数：10


In [2]:
# 初始化并训练模型
model = RandomForestClassifier(
    n_estimators=50,    # 决策树数量
    random_state=42,    # 固定随机种子
    n_jobs=-1           # 利用全部CPU核心加速
)
model.fit(X_train, y_train)
print("模型训练完成！")

# 测试集推理并计算准确率
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"=== 模型效果 ===")
print(f"手写数字分类准确率：{acc:.4f}（满分1.0）")


模型训练完成！
=== 模型效果 ===
手写数字分类准确率：0.9669（满分1.0）


In [3]:
# 1. 输出分类详细报告（每类数字的精准率、召回率）
from sklearn.metrics import classification_report
print("【分类详细报告】")
print(classification_report(
    y_test, y_pred,
    target_names=[f"数字{i}" for i in range(10)],
    digits=4  # 保留4位小数
))

# 2. 统计预测正确/错误的样本数
correct_num = sum(y_test == y_pred)
wrong_num = len(y_test) - correct_num
print(f"\n【整体统计】")
print(f"测试集总样本数：{len(y_test)}")
print(f"预测正确样本数：{correct_num}（占比{correct_num/len(y_test):.4f}）")
print(f"预测错误样本数：{wrong_num}（占比{wrong_num/len(y_test):.4f}）")

# 3. 随机展示10个样本的真实/预测结果
print(f"\n【随机10个样本验证】")
for i in range(10):
    idx = np.random.randint(0, len(y_test))
    print(f"  样本{idx}：真实数字={y_test[idx]}，预测数字={y_pred[idx]}（{'正确' if y_test[idx]==y_pred[idx] else '错误'}）")

print("\n=== 步骤三完成 ===")


【分类详细报告】
              precision    recall  f1-score   support

         数字0     0.9662    0.9908    0.9783       980
         数字1     0.9877    0.9921    0.9899      1135
         数字2     0.9539    0.9622    0.9580      1032
         数字3     0.9499    0.9574    0.9536      1010
         数字4     0.9734    0.9684    0.9709       982
         数字5     0.9740    0.9675    0.9708       892
         数字6     0.9750    0.9770    0.9760       958
         数字7     0.9723    0.9553    0.9637      1028
         数字8     0.9584    0.9466    0.9525       974
         数字9     0.9570    0.9495    0.9532      1009

    accuracy                         0.9669     10000
   macro avg     0.9668    0.9667    0.9667     10000
weighted avg     0.9669    0.9669    0.9669     10000


【整体统计】
测试集总样本数：10000
预测正确样本数：9669（占比0.9669）
预测错误样本数：331（占比0.0331）

【随机10个样本验证】
  样本6028：真实数字=5，预测数字=5（正确）
  样本3770：真实数字=4，预测数字=4（正确）
  样本9686：真实数字=2，预测数字=2（正确）
  样本7610：真实数字=3，预测数字=3（正确）
  样本6399：真实数字=6，预测数字=6（正确）
  样本7560：真实数字=4，预